# Observabilidad y Evolución de Arquitectura de Datos (Aplicado)

## Objetivos
- Implementar stack completo de observabilidad (Prometheus, Grafana, ELK).
- Diseñar sistema de monitoreo para pipelines de datos.
- Medir y gestionar deuda técnica con métricas cuantitativas.
- Aplicar estrategias de refactoring sin romper producción.
- Implementar versionado de schemas y migraciones seguras.
- Planificar evolución incremental de arquitectura.

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Arquitectura de Datos como Habilitador de Cultura Data-Driven](13_arquitectura_habilitadora_cultura_data_driven.ipynb) | Siguiente: [Arquitectura AI-Ready: Feature Stores y Streaming para ML](15_arquitectura_ai_ready_feature_stores_streaming.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. Objetivos
2. 1. Stack de Observabilidad para Datos
3. 2. Métricas Operacionales para Pipelines
4. 3. Código: Sistema de Monitoreo de Pipelines
5. 4. Gestión de Deuda Técnica
6. 5. Estrategias de Refactoring Seguro
7. 6. Versionado de Schemas
8. 7. Código: Simulador de Versionado de Schema

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Observabilidad y Evolución de Arquitectura de Datos (Aplicado)"] --> C["Notebook siguiente"]
```


## 1. Stack de Observabilidad para Datos

### Componentes
```mermaid
graph TD
    Pipelines[Data Pipelines] -->|Logs| Fluentd
    Pipelines -->|Metrics| Prometheus
    Pipelines -->|Traces| Jaeger
    
    Fluentd --> Elasticsearch
    Elasticsearch --> Kibana[Kibana Dashboards]
    
    Prometheus --> Grafana[Grafana Dashboards]
    Prometheus --> Alertmanager
    Alertmanager --> Slack
    Alertmanager --> PagerDuty
    
    Jaeger --> JaegerUI[Tracing UI]
    
    style Prometheus fill:#f9f
    style Grafana fill:#9f9
    style Elasticsearch fill:#bbf
```

### Roles por Herramienta
| Herramienta | Propósito | Ejemplo |
|-------------|-----------|--------|
| **Prometheus** | Recopilar métricas time-series | CPU, memoria, throughput, latencia |
| **Grafana** | Visualizar métricas en dashboards | Paneles con gráficos de línea, barras |
| **Elasticsearch** | Almacenar y buscar logs | "Buscar errores en pipeline X últimas 24h" |
| **Kibana** | Explorar logs con queries | Filtrar por severity, timestamp, componente |
| **Fluentd/Filebeat** | Recopilar logs de múltiples fuentes | Agregar logs de 10 servidores en un lugar |
| **Alertmanager** | Gestionar alertas y notificaciones | Deduplicar, agrupar, enrutar alertas |
| **Jaeger** | Distributed tracing | Seguir request desde API → Pipeline → DB |

## 2. Métricas Operacionales para Pipelines

### Golden Signals (Adaptado a Datos)
1. **Latency:** Tiempo desde evento hasta disponibilidad en capa Gold.
2. **Throughput:** Filas procesadas por segundo/minuto.
3. **Errors:** Tasa de fallos (%) en ingestión, transformación, calidad.
4. **Saturation:** Uso de recursos (CPU, memoria, disco, conexiones DB).

### Métricas Específicas de Datos
| Métrica | Definición | Umbral Típico |
|---------|-----------|---------------|
| **Data Freshness** | Tiempo desde última actualización | <1h para real-time, <24h batch |
| **Completeness** | % de registros sin nulos críticos | >99% |
| **Schema Drift** | Cambios no controlados en schema | 0 (detectar automáticamente) |
| **Duplicate Rate** | % de registros duplicados | <0.1% |
| **Row Count Variance** | Cambio vs promedio histórico | ±20% (alertar si >30%) |
| **Query Performance** | P95 latencia de queries | <5s para dashboards |
| **Cost per GB** | $ por GB procesado/almacenado | Benchmark vs industria |

## 3. Código: Sistema de Monitoreo de Pipelines

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import List, Dict, Optional
import random

@dataclass
class PipelineMetrics:
    pipeline_name: str
    timestamp: datetime
    duration_minutes: float
    rows_processed: int
    rows_failed: int
    data_quality_score: float  # 0-100
    cpu_avg_pct: float
    memory_avg_pct: float
    cost_usd: float
    
    @property
    def success_rate(self) -> float:
        total = self.rows_processed + self.rows_failed
        return (self.rows_processed / total * 100) if total > 0 else 0
    
    @property
    def throughput(self) -> float:
        """Filas por minuto."""
        return self.rows_processed / self.duration_minutes if self.duration_minutes > 0 else 0

class PipelineMonitor:
    def __init__(self, baseline_metrics: PipelineMetrics):
        self.baseline = baseline_metrics
        self.history: List[PipelineMetrics] = []
    
    def record(self, metrics: PipelineMetrics):
        self.history.append(metrics)
    
    def check_alerts(self, metrics: PipelineMetrics) -> List[Dict]:
        """Evalúa métricas actuales vs baseline y retorna alertas."""
        alerts = []
        
        # Alerta 1: Duración anormal
        if metrics.duration_minutes > self.baseline.duration_minutes * 1.5:
            alerts.append({
                "severity": "WARNING",
                "metric": "duration",
                "message": f"Pipeline {metrics.pipeline_name} tardó {metrics.duration_minutes:.1f}min (baseline: {self.baseline.duration_minutes:.1f}min)"
            })
        
        # Alerta 2: Calidad baja
        if metrics.data_quality_score < 95:
            severity = "CRITICAL" if metrics.data_quality_score < 90 else "WARNING"
            alerts.append({
                "severity": severity,
                "metric": "quality",
                "message": f"Calidad de datos {metrics.data_quality_score:.1f}% (objetivo: >95%)"
            })
        
        # Alerta 3: Tasa de error alta
        if metrics.rows_failed > 0:
            error_rate = metrics.rows_failed / (metrics.rows_processed + metrics.rows_failed) * 100
            if error_rate > 1:
                alerts.append({
                    "severity": "CRITICAL",
                    "metric": "error_rate",
                    "message": f"Tasa de error {error_rate:.2f}% ({metrics.rows_failed} filas fallaron)"
                })
        
        # Alerta 4: Recursos saturados
        if metrics.cpu_avg_pct > 80 or metrics.memory_avg_pct > 80:
            alerts.append({
                "severity": "WARNING",
                "metric": "resources",
                "message": f"Recursos altos: CPU {metrics.cpu_avg_pct:.1f}%, Mem {metrics.memory_avg_pct:.1f}%"
            })
        
        # Alerta 5: Costo anormal
        if metrics.cost_usd > self.baseline.cost_usd * 1.3:
            alerts.append({
                "severity": "WARNING",
                "metric": "cost",
                "message": f"Costo ${metrics.cost_usd:.2f} (+{(metrics.cost_usd/self.baseline.cost_usd-1)*100:.0f}% vs baseline)"
            })
        
        return alerts
    
    def generate_report(self) -> str:
        """Resumen de últimas 7 ejecuciones."""
        if not self.history:
            return "No hay datos"
        
        recent = self.history[-7:]
        avg_duration = sum(m.duration_minutes for m in recent) / len(recent)
        avg_quality = sum(m.data_quality_score for m in recent) / len(recent)
        total_rows = sum(m.rows_processed for m in recent)
        total_cost = sum(m.cost_usd for m in recent)
        
        return f"""Pipeline: {self.baseline.pipeline_name}
Últimas {len(recent)} ejecuciones:
  Duración promedio:  {avg_duration:.1f} min
  Calidad promedio:   {avg_quality:.1f}%
  Filas totales:      {total_rows:,}
  Costo total:        ${total_cost:.2f}
"""

# Ejemplo: Monitorear pipeline
baseline = PipelineMetrics(
    pipeline_name="sales_daily",
    timestamp=datetime.now() - timedelta(days=30),
    duration_minutes=12.0,
    rows_processed=1_200_000,
    rows_failed=0,
    data_quality_score=98.5,
    cpu_avg_pct=45,
    memory_avg_pct=60,
    cost_usd=8.50
)

monitor = PipelineMonitor(baseline)

# Simulación: Ejecución con problemas
current = PipelineMetrics(
    pipeline_name="sales_daily",
    timestamp=datetime.now(),
    duration_minutes=22.0,  # Lento
    rows_processed=1_180_000,
    rows_failed=25_000,  # Errores
    data_quality_score=92.0,  # Calidad baja
    cpu_avg_pct=85,  # Saturado
    memory_avg_pct=78,
    cost_usd=12.50  # Costo alto
)

monitor.record(current)
alerts = monitor.check_alerts(current)

print("=== Monitoreo de Pipeline ===")
print(f"Pipeline: {current.pipeline_name}")
print(f"Timestamp: {current.timestamp.strftime('%Y-%m-%d %H:%M')}")
print(f"Duración: {current.duration_minutes} min")
print(f"Throughput: {current.throughput:,.0f} filas/min")
print(f"Tasa de éxito: {current.success_rate:.2f}%")
print(f"Calidad: {current.data_quality_score}%")
print()

if alerts:
    print(f"⚠️  {len(alerts)} ALERTAS DETECTADAS:")
    for alert in alerts:
        icon = "🔴" if alert['severity'] == "CRITICAL" else "⚠️"
        print(f"  {icon} [{alert['severity']}] {alert['message']}")
else:
    print("✓ Sin alertas")

## 4. Gestión de Deuda Técnica

### Medición de Deuda Técnica
**Métricas:**
1. **Code Complexity:** Cyclomatic complexity (ramas condicionales).
2. **Test Coverage:** % de código con tests (objetivo: >80%).
3. **Documentation Coverage:** % de funciones/pipelines documentados.
4. **Dependency Age:** Días desde última actualización de librerías.
5. **TODO Density:** Número de comentarios `# TODO` o `# FIXME` por 1000 líneas.

### Priorización: Matriz Impacto vs Esfuerzo
```
  Impacto Alto
      ▲
      │
  [1] │ [2]
      │
──────┼──────► Esfuerzo Bajo
      │
  [3] │ [4]
      │
  Impacto Bajo

[1] Alto impacto, bajo esfuerzo → HACER YA (quick wins)
[2] Alto impacto, alto esfuerzo → PLANIFICAR (roadmap)
[3] Bajo impacto, bajo esfuerzo → HACER si hay tiempo
[4] Bajo impacto, alto esfuerzo → IGNORAR (no vale la pena)
```

### Ejemplo de Priorización
| Deuda Técnica | Impacto | Esfuerzo | Cuadrante | Acción |
|---------------|---------|----------|-----------|--------|
| Credenciales hardcodeadas | Alto | Bajo | [1] | Sprint actual |
| Migrar de Hadoop a Spark | Alto | Alto | [2] | Roadmap Q2 |
| Refactor función util_parse() | Medio | Bajo | [3] | Backlog |
| Reescribir pipeline legacy completo | Alto | Muy Alto | [2] | Roadmap Q3-Q4 |
| Formatear código (linting) | Bajo | Bajo | [3] | Automatizar CI/CD |

## 5. Estrategias de Refactoring Seguro

### 1. Strangler Fig Pattern
**Concepto:** Reemplazar sistema viejo gradualmente (como higuera estranguladora).

**Pasos:**
1. Crear nuevo pipeline paralelo al viejo.
2. Redirigir 10% tráfico al nuevo (shadow mode).
3. Comparar outputs (viejo vs nuevo) → validar equivalencia.
4. Incrementar tráfico gradualmente: 10% → 50% → 100%.
5. Deprecar pipeline viejo.

**Ventaja:** Rollback fácil si nuevo pipeline falla.

---

### 2. Blue-Green Deployment
**Concepto:** Dos ambientes idénticos (blue=producción, green=nueva versión).

**Pasos:**
1. Desplegar nueva versión en ambiente green.
2. Ejecutar tests en green.
3. Cambiar tráfico de blue → green (switch instantáneo).
4. Monitorear green; si hay problemas, switch back a blue.

**Ventaja:** Downtime cero, rollback inmediato.

---

### 3. Feature Flags
**Concepto:** Activar/desactivar features sin redesplegar.

**Ejemplo:**
```python
if feature_flag('use_new_validation'):
    result = new_validation_logic(data)
else:
    result = old_validation_logic(data)
```

**Ventaja:** Probar feature con 5% usuarios antes de 100%.

---

### 4. Expand-Contract Pattern
**Para cambios de schema.**

**Fase 1: Expand (agregar columna nueva)**
- Añadir `customer_email_v2` sin eliminar `customer_email`.
- Pipelines nuevos escriben ambas columnas.
- Consumidores migran gradualmente a `_v2`.

**Fase 2: Contract (eliminar columna vieja)**
- Después de 30 días, todos los consumidores usan `_v2`.
- Eliminar `customer_email` (ahora sin usar).

## 6. Versionado de Schemas

### Problema
**Sin versionado:**
- Productor cambia schema → consumidores se rompen sin aviso.
- Imposible hacer rollback (datos ya transformados).

### Solución: Schema Registry
**Herramientas:** Confluent Schema Registry, AWS Glue Schema Registry.

**Workflow:**
1. Productor registra schema v1.0 en registry.
2. Consumidores leen schema desde registry.
3. Productor propone cambio → schema v2.0.
4. Registry valida compatibilidad (backward/forward).
5. Si compatible, aprueba v2.0; si no, rechaza.

### Tipos de Compatibilidad
| Tipo | Cambios Permitidos | Ejemplo |
|------|-------------------|--------|
| **Backward** | Consumidores viejos leen datos nuevos | Agregar campo opcional |
| **Forward** | Consumidores nuevos leen datos viejos | Eliminar campo opcional |
| **Full** | Ambos (backward + forward) | Agregar campo opcional con default |
| **None** | Sin garantías (breaking change) | Cambiar tipo de campo |

## 7. Código: Simulador de Versionado de Schema

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional
from enum import Enum

class FieldType(Enum):
    STRING = "string"
    INT = "int"
    FLOAT = "float"
    BOOLEAN = "boolean"

@dataclass
class Field:
    name: str
    type: FieldType
    required: bool = True
    default: Optional[any] = None

@dataclass
class Schema:
    version: str
    fields: List[Field]
    
    def field_names(self) -> set:
        return {f.name for f in self.fields}

class SchemaRegistry:
    def __init__(self):
        self.schemas: Dict[str, List[Schema]] = {}  # dataset_name -> [schemas]
    
    def register(self, dataset_name: str, schema: Schema) -> bool:
        """Registra schema si es compatible con última versión."""
        if dataset_name not in self.schemas:
            self.schemas[dataset_name] = [schema]
            return True
        
        last_schema = self.schemas[dataset_name][-1]
        
        if self._is_backward_compatible(last_schema, schema):
            self.schemas[dataset_name].append(schema)
            return True
        else:
            return False
    
    def _is_backward_compatible(self, old: Schema, new: Schema) -> bool:
        """Backward: consumidores viejos pueden leer datos nuevos.
        Permitido: agregar campos opcionales, eliminar campos opcionales.
        Prohibido: eliminar campos requeridos, cambiar tipo.
        """
        old_fields = {f.name: f for f in old.fields}
        new_fields = {f.name: f for f in new.fields}
        
        # Validar campos requeridos del schema viejo
        for field_name, field in old_fields.items():
            if field.required:
                if field_name not in new_fields:
                    print(f"  ✗ Breaking: campo requerido '{field_name}' eliminado")
                    return False
                if new_fields[field_name].type != field.type:
                    print(f"  ✗ Breaking: tipo de '{field_name}' cambió")
                    return False
        
        # Validar nuevos campos
        for field_name, field in new_fields.items():
            if field_name not in old_fields:
                if field.required and field.default is None:
                    print(f"  ✗ Breaking: nuevo campo '{field_name}' es requerido sin default")
                    return False
        
        return True
    
    def get_latest(self, dataset_name: str) -> Optional[Schema]:
        if dataset_name in self.schemas:
            return self.schemas[dataset_name][-1]
        return None

# Ejemplo: Evolución de schema
registry = SchemaRegistry()

# Schema v1.0
schema_v1 = Schema(
    version="1.0",
    fields=[
        Field("customer_id", FieldType.INT, required=True),
        Field("email", FieldType.STRING, required=True),
        Field("created_at", FieldType.STRING, required=True)
    ]
)

print("=== Registro de Schemas ===")
print(f"\nRegistrando schema v1.0...")
registry.register("customers", schema_v1)
print("✓ v1.0 registrado (primera versión)")

# Schema v2.0: Agregar campo opcional (compatible)
schema_v2 = Schema(
    version="2.0",
    fields=[
        Field("customer_id", FieldType.INT, required=True),
        Field("email", FieldType.STRING, required=True),
        Field("created_at", FieldType.STRING, required=True),
        Field("phone", FieldType.STRING, required=False, default="")  # Nuevo, opcional
    ]
)

print(f"\nRegistrando schema v2.0 (agregar campo opcional)...")
if registry.register("customers", schema_v2):
    print("✓ v2.0 registrado (backward compatible)")
else:
    print("✗ v2.0 rechazado (breaking change)")

# Schema v3.0: Eliminar campo requerido (NO compatible)
schema_v3_bad = Schema(
    version="3.0",
    fields=[
        Field("customer_id", FieldType.INT, required=True),
        # email eliminado (era requerido)
        Field("created_at", FieldType.STRING, required=True),
        Field("phone", FieldType.STRING, required=False, default="")
    ]
)

print(f"\nRegistrando schema v3.0 (eliminar campo requerido)...")
if registry.register("customers", schema_v3_bad):
    print("✓ v3.0 registrado")
else:
    print("✗ v3.0 rechazado (breaking change detectado)")

print(f"\nSchemas registrados para 'customers': {[s.version for s in registry.schemas['customers']]}")

## 8. Planificación de Evolución Incremental

### Principios
1. **No big bang:** Cambios graduales (1-2 semanas) en lugar de reescritura completa (6 meses).
2. **Reversible:** Siempre poder hacer rollback.
3. **Monitoreado:** Comparar métricas antes/después.
4. **Comunicado:** Stakeholders informados de cambios.

### Roadmap de Evolución (Ejemplo)
| Trimestre | Iniciativa | Objetivo | Métricas de Éxito |
|-----------|-----------|----------|------------------|
| **Q1** | Migrar logs a ELK | Centralizar observabilidad | 100% pipelines logueando, MTTD -50% |
| **Q2** | Implementar Schema Registry | Prevenir breaking changes | 0 incidentes por cambios de schema |
| **Q3** | Refactor pipeline crítico (strangler) | Reducir deuda técnica | Latencia -30%, costo -20% |
| **Q4** | Automatizar tests de calidad | Prevenir data quality issues | 90% cobertura, incidentes -60% |

### Checklist Pre-Cambio
- [ ] Código revisado por peer review.
- [ ] Tests automatizados pasan (unit + integration).
- [ ] Documentación actualizada (README, ADR).
- [ ] Rollback plan definido.
- [ ] Stakeholders notificados (si cambio visible).
- [ ] Métricas baseline capturadas (para comparar después).
- [ ] Despliegue en ambiente staging validado.
- [ ] Monitoreo activo durante 24h post-despliegue.

## 9. Ejercicio Integrador
**Contexto:** Tu equipo tiene 5 pipelines en producción. Quieres implementar observabilidad completa.

**Tareas:**
1. Diseña stack de observabilidad (herramientas + flujo de datos).
2. Define 10 métricas clave a monitorear (2 por Golden Signal).
3. Configura 5 alertas críticas con umbrales específicos.
4. Identifica 3 items de deuda técnica y priorízalos (matriz impacto/esfuerzo).
5. Propón estrategia de refactoring para pipeline legacy (strangler, blue-green, o feature flags).
6. Diseña proceso de versionado de schemas con 3 escenarios de cambio.

## 10. Resumen
- **Stack de observabilidad:** Prometheus/Grafana (métricas), ELK (logs), Jaeger (tracing).
- **Golden Signals para datos:** Latency, Throughput, Errors, Saturation + métricas específicas (freshness, quality).
- **Deuda técnica cuantificable:** Complejidad, cobertura, TODOs, edad de dependencias.
- **Refactoring seguro:** Strangler Fig, Blue-Green, Feature Flags, Expand-Contract.
- **Versionado de schemas:** Schema Registry con validación de compatibilidad (backward/forward).
- **Evolución incremental:** Cambios graduales, reversibles, monitoreados.

**Siguiente (Senior):** Estrategia de evolución arquitectónica, fitness functions, modernización vs reemplazo.

<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- Referencias del curso: ../04-recursos/referencias.md
